In [67]:
import requests
from bs4 import BeautifulSoup
import csv
import re
import time


In [75]:
def scrape_unirun_results(base_url, page_count: int = 190, parcial: str = "K2_5"):
    all_results = []
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    is_first_url = True
    for i in range(page_count):
        url = base_url + f"?ordre=finish_time&page={i+1}"
        print(f"Scraping: {url}")
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Failed to retrieve {url}")
            continue
            
        soup = BeautifulSoup(response.content, 'html.parser')
        table = soup.find('table', id='resultats')
        
        # Find all main result rows
        rows = table.find_all('tr', class_='resultrow')

        for row in rows:
            data = {}
            if is_first_url:
                #data['first_row'] = row
                is_first_url = False
            # 1. Basic Info from main row
            data['pos_gen'] = row.find('td', class_='row1').get_text(strip=True)
            data['pos_sex'] = row.find('td', class_='row2').get_text(strip=True)
            data['bib'] = row.find('td', class_='row3').get_text(strip=True)
            data['name'] = row.find('td', class_='row4').get_text(strip=True)
            
            # Times are usually in row5
            times_td = row.find('td', class_='row5')
            if times_td:
                # Official time is often the first span, Real time is in the (tr ...) section
                data['time_official'] = times_td.find('span', id=lambda x: x and 'temps_actual_oficial' in x).get_text(strip=True) if times_td.find('span', id=lambda x: x and 'temps_actual_oficial' in x) else ""
                data['time_real'] = times_td.find('span', id=lambda x: x and 'temps_actual' in x and 'oficial' not in x).get_text(strip=True) if times_td.find('span', id=lambda x: x and 'temps_actual' in x and 'oficial' not in x) else ""

            """# 2. Detailed Info from the NEXT sibling row (the collapsible one)
            detail_row = row.find_next_sibling('tr')
            if detail_row:
                # Category
                cat_span = detail_row.find('span', class_='category-name')
                data['category'] = cat_span.get_text(strip=True) if cat_span else ""
                
                # Category Position
                cat_pos_span = detail_row.find('span', class_='category-pos')
                data['pos_cat'] = cat_pos_span.get_text(strip=True) if cat_pos_span else ""
                
                # Pace (Ritme)
                pace_p = detail_row.find('p', id=lambda x: x and 'ritme' in x)
                if pace_p:
                    data['pace'] = pace_p.get_text(strip=True).replace('Ritme:', '').strip()
                
                # Club
                club_field = detail_row.find('div', class_='club-field')
                if club_field:
                    data['club'] = club_field.find('p').get_text(strip=True).replace('Club:', '').strip()"""

            # --- 2. Data from Collapsible Detail Row ---
            detail_row = row.find_next_sibling('tr', class_='bottom-border')
            if detail_row:
                # SEX (Gender)
                # Looks for the paragraph containing "Sexe:"
                sex_p = detail_row.find('p', string=re.compile(r'Sexe:', re.I))
                if not sex_p: # Fallback if text is inside strong tag
                    for p in detail_row.find_all('p'):
                        if 'Sexe:' in p.get_text():
                            sex_p = p
                            break
                data['sexe'] = sex_p.get_text(strip=True).replace('Sexe:', '').strip() if sex_p else ""

                # Category and Club
                cat_span = detail_row.find('span', class_='category-name')
                data['category'] = cat_span.get_text(strip=True) if cat_span else ""
                
                cat_pos = detail_row.find('span', class_='category-pos')
                data['pos_cat'] = cat_pos.get_text(strip=True) if cat_pos else ""
                
                club_div = detail_row.find('div', class_='club-field')
                data['club'] = club_div.get_text(strip=True).replace('Club:', '').strip() if club_div else ""

                # Pace (Ritme)
                pace_p = detail_row.find('p', id=lambda x: x and 'ritme' in x)
                data['pace'] = pace_p.get_text(strip=True).replace('Ritme:', '').strip() if pace_p else ""

                # Parcial Splits
                parcial_div = detail_row.find('div', parcial='parcial_temps_01')
                if parcial_div:
                    # Get all text inside the div and clean up extra whitespace/newlines
                    # separator=' ' helps ensure "Pos K2_5" and ":" don't get merged weirdly
                    div_text = parcial_div.get_text(separator=' ', strip=True)
                    
                    # 1. Extract K2.5 Time
                    # Matches "K2_5" followed by ":" and then the time (digits and colons)
                    pattern = rf'{re.escape(parcial)}\s*:\s*([\d:]+)'
                    time_match = re.search(pattern, div_text)
                    if time_match:
                        data[parcial+"_time"] = time_match.group(1)
                    
                    # 2. Extract K2.5 Position
                    # Matches "Pos K2_5" followed by ":" and digits
                    pattern = rf'Pos {re.escape(parcial)}\s*:\s*(\d+)'
                    pos_match = re.search(pattern, div_text)
                    if pos_match:
                        data[parcial+"_pos"] = pos_match.group(1)


            all_results.append(data)
        time.sleep(1)  # Be polite and avoid overwhelming the server

    return all_results


In [ ]:
base_url = "https://xipgroc.cat/ca/curses/UNIRUN2026/5k/resultats"
# Define your URLs
pages = {
    "2026": {"end_page": 190, "rounded_end_page": 190, "url": "5k/resultats", "parcial": "K2_5"},
    "2025": {"end_page": 150, "rounded_end_page": 150, "url": "5k/resultats", "parcial": "K2_5"},
    "2024": {"end_page": 120, "rounded_end_page": 120, "url": "5k/resultats", "parcial": "K2_5"},
    "2023": {"end_page": 140, "rounded_end_page": 140, "url": "5k/resultats", "parcial": "K4"},
    "2022": {"end_page": 130, "rounded_end_page": 130, "url": "5k/resultats", "parcial": "K4"},
    "2021": {"end_page": 90, "rounded_end_page": 90, "url": "5k/resultats", "parcial": "K4"},
    "2020": {"end_page": 135, "rounded_end_page": 140, "url": "6779m/resultats", "parcial": "k5"},
    "2019": {"end_page": 123, "rounded_end_page": 130, "url": "6779m/resultats", "parcial": "k5"},
    "2018": {"end_page": 119, "rounded_end_page": 120, "url": "6779m/resultats", "parcial": "k5"},
    "2017": {"end_page": 85, "rounded_end_page": 90, "url": "6779m/resultats", "parcial": "k5"}

}

for year, page_info in pages.items():
    end_page = page_info["end_page"]
    url_suffix = page_info["url"]
    parcial = page_info["parcial"]
    print(f"Scraping year {year} end page {end_page}")
    base_url = f"https://xipgroc.cat/ca/curses/UNIRUN{year}/{url_suffix}"
    results_year = scrape_unirun_results(base_url=base_url, page_count=end_page+1, parcial=parcial)
    print(f"Successfully scraped {len(results_year)} runners for year {year}")
    keys = results_year[0].keys()
    with open(f'unirun_{year}_results.csv', 'w', newline='', encoding='utf-8') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(results_year)




Scraping year 2022 end page 130
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=1
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=2
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=3
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=4
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=5
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=6
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=7
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=8
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=9
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_time&page=10
Scraping: https://xipgroc.cat/ca/curses/UNIRUN2022/5k/resultats?ordre=finish_